# SO-101 ACT Pick/Place with PhysicalAI Studio Export and OpenVINO Physical AI API

This notebook shows how to export a PhysicalAI Studio ACT policy to OpenVINO and run it with the PhysicalAI runtime API.

```text
PhysicalAI Studio ACT checkpoint (.ckpt)
    -> policy.export(..., backend="openvino")
    -> PhysicalAI policy package
    -> InferenceModel(OpenVINO backend)
    -> CPU/GPU deployment benchmark
    -> local SO-101 replay visualization
```

The example uses:

- a PhysicalAI Studio / Lightning ACT checkpoint (`.ckpt`)
- a LeRobot-style SO-101 replay dataset with two cameras (`top_cam` and `arm_cam`)
- PhysicalAI Studio export APIs for OpenVINO model export
- PhysicalAI Runtime APIs for loading, resetting, and selecting actions


## Prerequisites

Install the required packages and clone the PhysicalAI Runtime and Physical AI Studio repositories.


In [ ]:
from pathlib import Path

requirements_file = Path("requirements.txt")
if not requirements_file.exists():
    requirements_file = Path("notebooks/requirements.txt")

if not Path("physicalai").exists():
    !git clone --depth 1 https://github.com/openvinotoolkit/physicalai.git
if not Path("physical-ai-studio").exists():
    !git clone --depth 1 https://github.com/open-edge-platform/physical-ai-studio.git

%pip install -q --extra-index-url https://download.pytorch.org/whl/cpu -r {requirements_file}
%pip install -q -e physicalai
%pip install -q -e "./physical-ai-studio/library[cpu]"


## 1. Configure the Notebook

Set paths, initialize OpenVINO Runtime, and define the SO-101 joint order used in the replay visualization.

By default, the notebook downloads the validation checkpoint and replay dataset from Hugging Face:

- checkpoint: [`yoyowz/so101-act-2cam-pick-place`](https://huggingface.co/yoyowz/so101-act-2cam-pick-place)
- replay dataset: [`yoyowz/so101-2cam-pick-place-validation`](https://huggingface.co/datasets/yoyowz/so101-2cam-pick-place-validation)

You can still override local paths with `PHYSICALAI_ACT_CHECKPOINT`, `PHYSICALAI_REPLAY_DATASET`, and `PHYSICALAI_REPLAY_EPISODE`.


In [ ]:
from pathlib import Path
import json
import os
import time

import numpy as np
import openvino as ov
from huggingface_hub import hf_hub_download, snapshot_download
from physicalai.inference import InferenceModel

WORKSPACE = Path.cwd().resolve()
RUNTIME_ROOT = WORKSPACE / "physicalai"
ROOT = RUNTIME_ROOT
os.chdir(ROOT)

MODEL_REPO_ID = "yoyowz/so101-act-2cam-pick-place"
CHECKPOINT_FILENAME = "2cam_angle_t50_3.ckpt"
DATASET_REPO_ID = "yoyowz/so101-2cam-pick-place-validation"
ASSETS_DIR = Path(os.environ.get("PHYSICALAI_ASSETS_DIR", WORKSPACE / "physicalai_assets")).resolve()

CHECKPOINT_PATH = Path(os.environ["PHYSICALAI_ACT_CHECKPOINT"]).resolve() if os.environ.get("PHYSICALAI_ACT_CHECKPOINT") else None
REPLAY_DATASET_DIR = Path(os.environ["PHYSICALAI_REPLAY_DATASET"]).resolve() if os.environ.get("PHYSICALAI_REPLAY_DATASET") else None
REPLAY_EPISODE_ID = int(os.environ.get("PHYSICALAI_REPLAY_EPISODE", "0"))
EXPORT_NAME = os.environ.get("PHYSICALAI_EXPORT_NAME", f"{Path(CHECKPOINT_FILENAME).stem}_openvino")
EXPORT_DIR = ROOT / "exports" / EXPORT_NAME
CACHE_DIR = ROOT / "exports" / f"{EXPORT_NAME}_cache"
VIS_DIR = ROOT / "exports" / "so101_visualization"
GPU_PRECISION_HINT = os.environ.get("PHYSICALAI_GPU_PRECISION_HINT", "f32").strip().lower()

SO101_JOINT_ORDER = (
    "shoulder_pan",
    "shoulder_lift",
    "elbow_flex",
    "wrist_flex",
    "wrist_roll",
    "gripper",
)

core = ov.Core()
print("[INFO] OpenVINO:", ov.__version__)
print("[INFO] Available devices:", core.available_devices)
print("[INFO] Checkpoint repo:", MODEL_REPO_ID)
print("[INFO] Replay dataset repo:", DATASET_REPO_ID, "episode:", REPLAY_EPISODE_ID)
print("[INFO] Export directory:", EXPORT_DIR)


## 2. Load a PhysicalAI Studio ACT Checkpoint

PhysicalAI Studio stores trained policies as Lightning checkpoints (`.ckpt`). Loading this format with `ACT.load_from_checkpoint()` restores the policy architecture, learned weights, and training-time metadata needed for export.

The export API is intentionally compact. Once the checkpoint is loaded, OpenVINO export is just a few lines:

```python
from physicalai.export import get_available_backends
from physicalai.policies import ACT

print(get_available_backends())
policy = ACT.load_from_checkpoint("path/to/policy.ckpt", map_location="cpu")
policy.export("./exports/act_policy", backend="openvino")
```

This avoids hand-written OpenVINO conversion code and lets PhysicalAI Studio create the runtime package manifest for PhysicalAI Runtime.


In [ ]:
from physicalai.export import get_available_backends
from physicalai.policies import ACT

print("[INFO] Available export backends:", get_available_backends())

if CHECKPOINT_PATH is None:
    CHECKPOINT_PATH = Path(hf_hub_download(
        repo_id=MODEL_REPO_ID,
        filename=CHECKPOINT_FILENAME,
        local_dir=ASSETS_DIR / "checkpoints",
        local_dir_use_symlinks=False,
    )).resolve()

if not CHECKPOINT_PATH.exists():
    raise FileNotFoundError(
        "PhysicalAI Studio export requires a Lightning .ckpt checkpoint. "
        f"Expected: {CHECKPOINT_PATH}. "
        "Set PHYSICALAI_ACT_CHECKPOINT to a local checkpoint path if you do not want to download from Hugging Face."
    )

policy = ACT.load_from_checkpoint(str(CHECKPOINT_PATH), map_location="cpu")
policy.eval()

print("[INFO] Loaded checkpoint:", CHECKPOINT_PATH)
print("[INFO] Parameters:", sum(p.numel() for p in policy.parameters()))
print("[INFO] Supported export backends:", [str(x) for x in policy.get_supported_export_backends()])


## 3. Prepare Export Sample Inputs

PhysicalAI Studio's `policy.export(..., backend="openvino")` can use the policy model's built-in `sample_input`. The notebook records those input names and shapes so the OpenVINO runtime benchmark and dataset replay feed observations with the same names as the exported package.


In [ ]:
import torch

export_sample = policy.model.sample_input
EXPORT_INPUT_KEYS = list(export_sample.keys())
STATE_INPUT_KEY = next((key for key in EXPORT_INPUT_KEYS if key == "state" or key.endswith("state")), None)
IMAGE_INPUT_KEYS = [key for key in EXPORT_INPUT_KEYS if "image" in key or key.startswith("images")]
FRONT_INPUT_KEY = next((key for key in IMAGE_INPUT_KEYS if "front" in key.lower()), IMAGE_INPUT_KEYS[0] if IMAGE_INPUT_KEYS else None)
HANDEYE_INPUT_KEY = next((key for key in IMAGE_INPUT_KEYS if "handeye" in key.lower() or "wrist" in key.lower()), None)
if HANDEYE_INPUT_KEY is None and len(IMAGE_INPUT_KEYS) > 1:
    HANDEYE_INPUT_KEY = next(key for key in IMAGE_INPUT_KEYS if key != FRONT_INPUT_KEY)

missing = [
    name for name, value in {
        "STATE_INPUT_KEY": STATE_INPUT_KEY,
        "FRONT_INPUT_KEY": FRONT_INPUT_KEY,
        "HANDEYE_INPUT_KEY": HANDEYE_INPUT_KEY,
    }.items() if value is None
]
if missing:
    raise RuntimeError(f"Cannot infer required SO-101 export inputs from {EXPORT_INPUT_KEYS}; missing {missing}")

state_t = export_sample[STATE_INPUT_KEY].detach().cpu().float()
front_t = export_sample[FRONT_INPUT_KEY].detach().cpu().float()
handeye_t = export_sample[HANDEYE_INPUT_KEY].detach().cpu().float()

STATE_SHAPE = tuple(state_t.shape[-1:])
FRONT_IMAGE_SHAPE = tuple(front_t.shape[-3:])
HANDEYE_IMAGE_SHAPE = tuple(handeye_t.shape[-3:])

print("[INFO] Export input keys:")
for key, value in export_sample.items():
    print(f"  - {key}: {tuple(value.shape)}")
print("[INFO] Selected SO-101 inputs:", {
    "state": STATE_INPUT_KEY,
    "front": FRONT_INPUT_KEY,
    "handeye": HANDEYE_INPUT_KEY,
})


## 4. Export a PhysicalAI Policy Package

`policy.export(..., backend="openvino")` creates the OpenVINO model artifact and a `manifest.json` file. The manifest is the contract used by PhysicalAI Runtime to identify the backend artifact, runner, and processing pipeline.


In [ ]:
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

print("[STEP] Exporting ACT policy with PhysicalAI Studio API ...")
policy.export(EXPORT_DIR, backend="openvino")

print("[DONE] PhysicalAI package ready:")
for path in sorted(EXPORT_DIR.glob("*")):
    if path.is_file():
        print(f"  - {path} ({path.stat().st_size / 1024 / 1024:.2f} MB)")


## 5. PhysicalAI Runtime: Load, Reset, Select Action

PhysicalAI Runtime provides a unified deployment API for exported policy packages. `InferenceModel.load()` reads the package manifest, loads the selected backend, and exposes policy-style methods for robot applications:

```python
model = InferenceModel.load("./exports/act_policy", backend="openvino", device="GPU")
model.reset()
action = model.select_action(observation)
```

`reset()` clears per-episode state such as queued ACT chunks. `select_action()` returns the next single robot action, while `predict_action_chunk()` returns the full predicted action chunk.


In [ ]:
handeye_np = handeye_t.numpy().astype(np.float32)
front_np = front_t.numpy().astype(np.float32)
state_np = state_t.numpy().astype(np.float32)

# Runtime helpers use the PhysicalAI API. The OpenVINO IR produced above is
# consumed by InferenceModel via manifest.json.
def make_policy_observation(handeye: np.ndarray, front: np.ndarray, state: np.ndarray) -> dict[str, np.ndarray]:
    return {
        HANDEYE_INPUT_KEY: np.asarray(handeye, dtype=np.float32),
        FRONT_INPUT_KEY: np.asarray(front, dtype=np.float32),
        STATE_INPUT_KEY: np.asarray(state, dtype=np.float32),
    }

def openvino_config_for_device(device: str) -> dict[str, str]:
    if device.startswith("GPU") and GPU_PRECISION_HINT not in {"", "none", "default"}:
        # Current Intel GPU plugin/driver combinations may require f32 for this ACT graph.
        return {"INFERENCE_PRECISION_HINT": GPU_PRECISION_HINT}
    return {}

def benchmark_physicalai(device: str, runs: int = 20, compile_config: dict[str, str] | None = None):
    CACHE_DIR.mkdir(parents=True, exist_ok=True)
    compile_config = openvino_config_for_device(device) if compile_config is None else compile_config
    if compile_config:
        print(f"[INFO] OpenVINO config for {device}: {compile_config}")
    start = time.perf_counter()
    model = InferenceModel.load(EXPORT_DIR, backend="openvino", device=device, CACHE_DIR=str(CACHE_DIR), **compile_config)
    load_ms = (time.perf_counter() - start) * 1000
    obs = make_policy_observation(handeye_np, front_np, state_np)
    model.reset()
    for _ in range(3):
        _ = model.predict_action_chunk(obs)
    first_action = model.select_action(obs)
    timings = []
    for _ in range(runs):
        start = time.perf_counter()
        chunk = model.predict_action_chunk(obs)
        timings.append((time.perf_counter() - start) * 1000)
    return model, {
        "device": device,
        "load_ms": load_ms,
        "avg_ms": float(np.mean(timings)),
        "p50_ms": float(np.percentile(timings, 50)),
        "p95_ms": float(np.percentile(timings, 95)),
        "fps": float(1000 / np.mean(timings)),
        "chunk_shape": tuple(chunk.shape),
        "select_action_shape": tuple(first_action.shape),
    }

def benchmark_physicalai_with_fallback(device: str, runs: int = 20):
    try:
        return benchmark_physicalai(device, runs=runs)
    except RuntimeError as exc:
        print(f"[WARN] PhysicalAI OpenVINO failed on {device}: {type(exc).__name__}: {exc}")
        if device != "CPU":
            print("[INFO] Falling back to CPU so the validation can continue.")
            return benchmark_physicalai("CPU", runs=runs)
        raise


## 6. Select Deployment Device

Choose the OpenVINO target device for the PhysicalAI runtime benchmark.


In [ ]:
import ipywidgets as widgets
from IPython.display import display

device_options = list(core.available_devices)
default_device = "GPU.1" if "GPU.1" in device_options else ("GPU" if "GPU" in device_options else "CPU")
TARGET_DEVICE = widgets.Dropdown(
    options=device_options,
    value=default_device if default_device in device_options else device_options[0],
    description="Device:",
)
display(TARGET_DEVICE)

In [ ]:
physicalai_model, selected_result = benchmark_physicalai_with_fallback(TARGET_DEVICE.value, runs=20)
print("[RESULT] PhysicalAI OpenVINO deployment")
for key, value in selected_result.items():
    print(f"  {key}: {value}")

## 7. Optional CPU/GPU Comparison

Run a short comparison against CPU after the selected-device benchmark.


In [ ]:
all_results = []

if "selected_result" in globals():
    all_results.append(selected_result)
    print(
        f"[OK] {selected_result['device']} (selected): "
        f"avg={selected_result['avg_ms']:.3f} ms, "
        f"p95={selected_result['p95_ms']:.3f} ms, "
        f"fps={selected_result['fps']:.2f}"
    )

if not any(res["device"] == "CPU" for res in all_results):
    try:
        _, cpu_res = benchmark_physicalai("CPU", runs=10)
        all_results.append(cpu_res)
        print(f"[OK] CPU: avg={cpu_res['avg_ms']:.3f} ms, p95={cpu_res['p95_ms']:.3f} ms, fps={cpu_res['fps']:.2f}")
    except Exception as exc:
        print(f"[WARN] CPU failed: {type(exc).__name__}: {exc}")

print("\n[SUMMARY]")
for res in all_results:
    print(
        f"{res['device']:>6} | load={res['load_ms']:.1f} ms | "
        f"avg={res['avg_ms']:.3f} ms | p95={res['p95_ms']:.3f} ms | fps={res['fps']:.2f}"
    )


## 8. Load Replay Episode

Load a LeRobot-style validation episode for replay visualization. By default, the dataset is downloaded from Hugging Face; if `PHYSICALAI_REPLAY_DATASET` is set, the notebook uses that local folder instead.

The loader reads episode metadata and seeks to the correct frame range inside the dataset video files.


In [ ]:
import pandas as pd
import cv2

EPISODE_ID = REPLAY_EPISODE_ID

if REPLAY_DATASET_DIR is None:
    REPLAY_DATASET_DIR = Path(snapshot_download(
        repo_id=DATASET_REPO_ID,
        repo_type="dataset",
        local_dir=ASSETS_DIR / "replay_dataset",
        local_dir_use_symlinks=False,
    )).resolve()

def dataset_video_key(model_input_key: str) -> str:
    if model_input_key.startswith("observation.images."):
        return model_input_key
    if model_input_key.startswith("images."):
        return "observation." + model_input_key
    return model_input_key

def load_episode_table(dataset_root: Path) -> pd.DataFrame:
    episode_files = sorted((dataset_root / "meta" / "episodes").glob("chunk-*/*.parquet"))
    if not episode_files:
        raise FileNotFoundError(f"No episode metadata files found under {dataset_root / 'meta' / 'episodes'}")
    return pd.concat([pd.read_parquet(path) for path in episode_files], ignore_index=True)

def load_local_episode(dataset_root: Path, episode_id: int):
    dataset_root = Path(dataset_root).resolve()
    info_path = dataset_root / "meta" / "info.json"
    if not info_path.exists():
        raise FileNotFoundError(f"Missing LeRobot dataset metadata: {info_path}")

    info = json.loads(info_path.read_text(encoding="utf-8"))
    episodes = load_episode_table(dataset_root)
    matches = episodes[episodes["episode_index"] == episode_id]
    if matches.empty:
        available = episodes["episode_index"].tolist()
        raise ValueError(f"Episode {episode_id} not found. Available episodes: {available[:10]} ... {available[-10:]}")
    episode_meta = matches.iloc[0]

    data_chunk = int(episode_meta["data/chunk_index"])
    data_file = int(episode_meta["data/file_index"])
    parquet = dataset_root / "data" / f"chunk-{data_chunk:03d}" / f"file-{data_file:03d}.parquet"
    episode_df = pd.read_parquet(parquet)
    episode_df = episode_df[episode_df["episode_index"] == episode_id].reset_index(drop=True)

    front_video_key = dataset_video_key(FRONT_INPUT_KEY)
    handeye_video_key = dataset_video_key(HANDEYE_INPUT_KEY)
    fps = float(info.get("fps", 30))

    def video_info(video_key: str):
        chunk_col = f"videos/{video_key}/chunk_index"
        file_col = f"videos/{video_key}/file_index"
        start_col = f"videos/{video_key}/from_timestamp"
        chunk_index = int(episode_meta[chunk_col]) if chunk_col in episode_meta.index else 0
        file_index = int(episode_meta[file_col]) if file_col in episode_meta.index else data_file
        start_seconds = float(episode_meta[start_col]) if start_col in episode_meta.index else 0.0
        video_path = dataset_root / "videos" / video_key / f"chunk-{chunk_index:03d}" / f"file-{file_index:03d}.mp4"
        start_frame = int(round(start_seconds * fps))
        return video_path, start_frame

    front_video, front_start = video_info(front_video_key)
    handeye_video, handeye_start = video_info(handeye_video_key)

    return episode_df, front_video, handeye_video, front_start, handeye_start, front_video_key, handeye_video_key, fps

episode_df, front_video_path, handeye_video_path, FRONT_VIDEO_START_FRAME, HANDEYE_VIDEO_START_FRAME, FRONT_DATASET_VIDEO_KEY, HANDEYE_DATASET_VIDEO_KEY, REPLAY_FPS = load_local_episode(REPLAY_DATASET_DIR, EPISODE_ID)
FRONT_CAMERA_LABEL = FRONT_DATASET_VIDEO_KEY.split(".")[-1]
HANDEYE_CAMERA_LABEL = HANDEYE_DATASET_VIDEO_KEY.split(".")[-1]

print("[INFO] Replay dataset:", REPLAY_DATASET_DIR)
print("[INFO] Episode:", EPISODE_ID, "rows:", len(episode_df), "fps:", REPLAY_FPS)
print("[INFO] Data columns:", episode_df.columns.tolist())
print("[INFO] Front video:", front_video_path, "start_frame:", FRONT_VIDEO_START_FRAME)
print("[INFO] Handeye video:", handeye_video_path, "start_frame:", HANDEYE_VIDEO_START_FRAME)
print("[INFO] Camera labels:", FRONT_CAMERA_LABEL, HANDEYE_CAMERA_LABEL)
print(episode_df.head(2))


## 9. PhysicalAI Replay Visualization

Replay a recorded SO-101 episode and overlay the PhysicalAI/OpenVINO actions.

- Left: recorded camera views.
- Right: a joint-space sketch of the observed state and predicted target.
- Bottom: predicted action vs dataset expert action for each joint.

Replay MAE is an offline domain-match sanity check. It is useful for confirming that the exported policy behaves consistently on a recorded validation episode, but real task quality still requires closed-loop robot or simulator evaluation.


In [ ]:
from PIL import Image, ImageDraw
from IPython.display import Image as IPyImage, display as ipy_display

def read_video_rgb(path: Path, max_frames: int | None = None, start_frame: int = 0):
    cap = cv2.VideoCapture(str(path))
    if start_frame:
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(start_frame))
    frames = []
    while True:
        ok, frame = cap.read()
        if not ok:
            break
        frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        frames.append(frame)
        if max_frames is not None and len(frames) >= max_frames:
            break
    cap.release()
    return frames

def image_to_chw_float(frame_rgb, input_key: str):
    _, height, width = FRONT_IMAGE_SHAPE if input_key == FRONT_INPUT_KEY else HANDEYE_IMAGE_SHAPE
    if frame_rgb.shape[:2] != (height, width):
        frame_rgb = cv2.resize(frame_rgb, (width, height), interpolation=cv2.INTER_AREA)
    return np.transpose(frame_rgb.astype(np.float32) / 255.0, (2, 0, 1))[None, ...]

def as_action_vector(values, name="action"):
    arr = np.asarray(values, dtype=np.float32)
    if arr.ndim == 0:
        raise ValueError(f"Expected {name} to be a vector, got scalar value {arr!r}")
    if arr.ndim > 1:
        arr = arr.reshape(-1, arr.shape[-1])[0]
    arr = arr[: len(SO101_JOINT_ORDER)].astype(np.float32)
    if arr.shape[0] != len(SO101_JOINT_ORDER):
        raise ValueError(f"Expected {name} length {len(SO101_JOINT_ORDER)}, got shape {arr.shape}")
    return arr

def so101_points(values, origin=(590, 330), scale=1.0):
    vals = np.asarray(values, dtype=np.float32)
    lengths = np.array([70, 58, 48, 34], dtype=np.float32) * scale
    angles = np.deg2rad([
        -90 + vals[0] * 0.55,
        vals[1] * 0.45,
        vals[2] * 0.35,
        vals[3] * 0.25 + vals[4] * 0.08,
    ])
    pts = [np.array(origin, dtype=np.float32)]
    heading = 0.0
    for length, angle in zip(lengths, angles):
        heading += angle
        pts.append(pts[-1] + np.array([np.cos(heading), np.sin(heading)]) * length)
    return [(int(x), int(y)) for x, y in pts]

def draw_arm(draw, values, color, width=9):
    pts = so101_points(values)
    for a, b in zip(pts[:-1], pts[1:]):
        draw.line([a, b], fill=color, width=width)
        draw.line([a, b], fill=(245, 248, 250), width=max(2, width // 3))
    for p in pts:
        draw.ellipse([p[0] - 7, p[1] - 7, p[0] + 7, p[1] + 7], fill=(20, 24, 30), outline=color, width=3)
    gripper = float(np.asarray(values)[5])
    ee = pts[-1]
    span = int(8 + np.clip(gripper, 0, 100) * 0.12)
    draw.line([(ee[0] - span, ee[1] - 9), (ee[0] + span, ee[1] + 9)], fill=color, width=4)

def draw_bars(draw, x, y, width, height, pred, expert):
    joint_names = list(SO101_JOINT_ORDER)
    row_h = height // len(joint_names)
    center = x + width // 2
    draw.line([(center, y), (center, y + height)], fill=(80, 90, 102), width=1)
    for i, name in enumerate(joint_names):
        yy = y + i * row_h + 6
        draw.text((x, yy), name, fill=(220, 226, 234))
        for val, color, offset in [(expert[i], (95, 170, 255), 13), (pred[i], (255, 190, 85), 28)]:
            v = float(np.clip(val, -100, 100))
            bar = int((v / 100.0) * (width * 0.32))
            if bar >= 0:
                draw.rectangle([center, yy + offset, center + bar, yy + offset + 8], fill=color)
            else:
                draw.rectangle([center + bar, yy + offset, center, yy + offset + 8], fill=color)
        draw.text((x + width - 105, yy + 10), f"E {expert[i]:6.1f}", fill=(95, 170, 255))
        draw.text((x + width - 105, yy + 27), f"P {pred[i]:6.1f}", fill=(255, 190, 85))

def make_overlay_frame(front, handeye, state, pred, expert, frame_idx, latency_ms, device):
    canvas = Image.new("RGB", (1120, 720), (18, 22, 28))
    front_img = Image.fromarray(front).resize((512, 288))
    hand_img = Image.fromarray(handeye).resize((512, 288))
    draw_front = ImageDraw.Draw(front_img)
    draw_hand = ImageDraw.Draw(hand_img)
    draw_front.rectangle([0, 0, 96, 26], fill=(0, 0, 0))
    draw_front.text((8, 6), FRONT_CAMERA_LABEL, fill=(255, 255, 255))
    draw_hand.rectangle([0, 0, 118, 26], fill=(0, 0, 0))
    draw_hand.text((8, 6), HANDEYE_CAMERA_LABEL, fill=(255, 255, 255))
    canvas.paste(front_img, (20, 72))
    canvas.paste(hand_img, (20, 374))

    draw = ImageDraw.Draw(canvas)
    draw.text((20, 24), "SO-101 Pick/Place Replay: PhysicalAI + OpenVINO ACT", fill=(242, 246, 250))
    draw.text((20, 48), f"checkpoint={CHECKPOINT_PATH.name} | device={device} | frame={frame_idx:03d} | latency={latency_ms:.1f} ms", fill=(170, 184, 199))

    draw.rectangle([560, 72, 1098, 430], outline=(65, 76, 90), width=2)
    draw.text((580, 92), "Joint-space viewer", fill=(235, 241, 245))
    draw.text((580, 116), "blue: observed state   amber: PhysicalAI predicted target", fill=(170, 184, 199))
    draw_arm(draw, state, (95, 170, 255), width=11)
    draw_arm(draw, pred, (255, 190, 85), width=7)

    err = float(np.mean(np.abs(pred - expert)))
    draw.text((580, 398), f"mean |pred - expert| = {err:.2f}", fill=(235, 241, 245))

    draw.rectangle([560, 454, 1098, 704], outline=(65, 76, 90), width=2)
    draw.text((580, 468), "Action comparison in SO-101 normalized joint space", fill=(235, 241, 245))
    draw_bars(draw, 580, 500, 490, 186, pred, expert)
    return canvas

def describe_replay_mae(mean_mae: float) -> str:
    if mean_mae < 5.0:
        return "low offline MAE on this replay dataset; export/runtime and replay domain look consistent"
    if mean_mae < 15.0:
        return "moderate offline MAE; inspect the overlay and confirm this replay matches the training domain"
    return "high offline MAE; often expected when the checkpoint was trained on a different dataset, camera setup, task, or action distribution"

def run_replay_visualization(max_rendered_frames=140, render_stride=2):
    max_replay_steps = max_rendered_frames * render_stride
    front_frames = read_video_rgb(front_video_path, max_frames=max_replay_steps, start_frame=FRONT_VIDEO_START_FRAME)
    handeye_frames = read_video_rgb(handeye_video_path, max_frames=max_replay_steps, start_frame=HANDEYE_VIDEO_START_FRAME)
    n = min(len(front_frames), len(handeye_frames), len(episode_df), max_replay_steps)
    print("[INFO] Loaded frames:", len(front_frames), len(handeye_frames), "using", n)

    replay_device = globals().get(
        "REPLAY_DEVICE",
        selected_result["device"] if "selected_result" in globals() else TARGET_DEVICE.value,
    )
    try:
        test_obs = {
            HANDEYE_INPUT_KEY: image_to_chw_float(handeye_frames[0], HANDEYE_INPUT_KEY),
            FRONT_INPUT_KEY: image_to_chw_float(front_frames[0], FRONT_INPUT_KEY),
            STATE_INPUT_KEY: as_action_vector(episode_df.iloc[0]["observation.state"], name="observation.state")[None, :],
        }
        if (
            replay_device != "CPU"
            and "physicalai_model" in globals()
            and "selected_result" in globals()
            and selected_result["device"] == replay_device
        ):
            print(f"[INFO] Reusing selected {replay_device} model for replay visualization.")
            model = physicalai_model
        else:
            compile_config = openvino_config_for_device(replay_device)
            if compile_config:
                print(f"[INFO] OpenVINO config for replay on {replay_device}: {compile_config}")
            model = InferenceModel.load(
                EXPORT_DIR,
                backend="openvino",
                device=replay_device,
                CACHE_DIR=str(CACHE_DIR),
                **compile_config,
            )
        _ = model.predict_action_chunk(test_obs)
    except RuntimeError as exc:
        if replay_device != "CPU":
            print(f"[WARN] Replay device {replay_device} failed: {type(exc).__name__}: {exc}")
            print("[INFO] Falling back to CPU for replay visualization.")
            replay_device = "CPU"
            model = InferenceModel.load(EXPORT_DIR, backend="openvino", device=replay_device, CACHE_DIR=str(CACHE_DIR))
        else:
            raise
    model.reset()
    vis_frames = []
    latencies = []
    errors = []
    predictions = []
    expert_actions = []

    for frame_idx in range(n):
        state = as_action_vector(episode_df.iloc[frame_idx]["observation.state"], name="observation.state")
        expert = as_action_vector(episode_df.iloc[frame_idx]["action"], name="expert action")
        front = front_frames[frame_idx]
        handeye = handeye_frames[frame_idx]
        obs = {
            HANDEYE_INPUT_KEY: image_to_chw_float(handeye, HANDEYE_INPUT_KEY),
            FRONT_INPUT_KEY: image_to_chw_float(front, FRONT_INPUT_KEY),
            STATE_INPUT_KEY: state[None, :],
        }
        start = time.perf_counter()
        raw_pred = model.select_action(obs)
        pred = as_action_vector(raw_pred, name="predicted action")
        latency_ms = (time.perf_counter() - start) * 1000
        if frame_idx == 0:
            print("[INFO] Replay action shapes:", {
                "raw_pred": tuple(np.asarray(raw_pred).shape),
                "pred": tuple(pred.shape),
                "expert": tuple(expert.shape),
            })
        latencies.append(latency_ms)
        errors.append(float(np.mean(np.abs(pred - expert))))
        predictions.append(pred)
        expert_actions.append(expert)
        if frame_idx % render_stride == 0 and len(vis_frames) < max_rendered_frames:
            vis_frames.append(make_overlay_frame(front, handeye, state, pred, expert, frame_idx, latency_ms, replay_device))

    VIS_DIR.mkdir(parents=True, exist_ok=True)
    gif_path = VIS_DIR / "so101_pick_place_physicalai_openvino.gif"
    vis_frames[0].save(gif_path, save_all=True, append_images=vis_frames[1:], duration=66, loop=0)
    mean_mae = float(np.mean(errors))
    per_joint_mae = np.mean(np.abs(np.stack(predictions) - np.stack(expert_actions)), axis=0)
    print("[RESULT] Replay steps:", len(predictions))
    print("[RESULT] Rendered frames:", len(vis_frames))
    print("[RESULT] Avg select_action latency ms:", float(np.mean(latencies)))
    print("[RESULT] Avg MAE vs expert action:", mean_mae)
    print("[RESULT] Per-joint MAE:", dict(zip(SO101_JOINT_ORDER, per_joint_mae.round(3).tolist())))
    print("[INTERPRETATION]", describe_replay_mae(mean_mae))
    print("[INTERPRETATION] Replay MAE is a domain-match sanity check, not an OpenVINO correctness test.")
    print("[INTERPRETATION] A large MAE can be normal when using a checkpoint trained on a different recording setup or task dataset.")
    print("[INTERPRETATION] For policy-quality validation, replay the training/validation dataset or run closed-loop robot evaluation.")
    print("[DONE] Saved GIF:", gif_path)
    ipy_display(IPyImage(filename=str(gif_path)))

run_replay_visualization(max_rendered_frames=140, render_stride=2)